In [43]:
import pandas as pd
import sys 

sys.path.append("..")  

%load_ext autoreload
%autoreload 2

from src.limpieza_datos import (
    detectar_duplicados_exactos,
    detectar_ids_duplicados,
    resolver_duplicados_por_id,
    corregir_edades_imposibles,
    normalizar_fechas,
    imputar_numericas_con_mediana,
    imputar_categorica_con_moda
)

In [44]:
df = pd.read_csv("../data/raw-data/pacientes.csv", sep=";")

print(df.shape)
df.head()

(515, 10)


,id_paciente,edad_paciente,sexo,peso_kg,altura_cm,fecha_registro,frecuencia_cardiaca_media_bpm,derivacion_ecg,frecuencia_muestreo_hz,etiqueta
0,P0305,82.0,M,69.8,168.0,4/10/2023,69.4,II,250,Normal
1,P0500,58.0,M,70.9,178.9,23/04/2023,79.2,II,250,Normal
2,P0442,49.0,M,84.2,173.1,25/01/2023,72.7,II,250,Normal
3,P0154,39.0,F,80.5,156.4,24/06/2023,87.0,II,250,Anormal
4,P0479,22.0,F,78.7,165.5,28/01/2023,77.8,II,250,Normal


In [45]:
df_clean = df.copy()

## Tratamiento de registros duplicados

Durante la exploración inicial se identificaron 15 registros duplicados.

Se conservará, para cada paciente repetido, el registro con menor cantidad de valores faltantes, priorizando la completitud de la información disponible.

In [46]:
# Aqui se aplican las funciones de limpieza de datos, explorar el archivo .py con la 
# documentación de cada función para entender qué hace cada una y cómo se aplican.

duplicados = detectar_duplicados_exactos(df_clean)

print(f"Filas duplicadas encontradas: {len(duplicados)}")

duplicados.head(10)

Filas duplicadas encontradas: 2


,id_paciente,edad_paciente,sexo,peso_kg,altura_cm,fecha_registro,frecuencia_cardiaca_media_bpm,derivacion_ecg,frecuencia_muestreo_hz,etiqueta
19,P0196,39.0,F,76.2,167.3,28/05/2023,69.8,II,250,Normal
245,P0196,39.0,F,76.2,167.3,28/05/2023,69.8,II,250,Normal


In [47]:
duplicados_por_id = detectar_ids_duplicados(df_clean, id_col="id_paciente")

print(f"IDs duplicados encontrados: {len(duplicados_por_id)}")

duplicados_por_id.head(30)

IDs duplicados encontrados: 30


,id_paciente,edad_paciente,sexo,peso_kg,altura_cm,fecha_registro,frecuencia_cardiaca_media_bpm,derivacion_ecg,frecuencia_muestreo_hz,etiqueta
33,P0010,56.0,F,78.3,165.5,14/06/2023,72.6,II,250,Normal
471,P0010,56.0,F,78.4,165.5,14/06/2023,72.6,II,250,Normal
507,P0022,39.0,F,62.0,155.0,15/10/2023,88.7,II,250,Normal
481,P0022,39.0,F,65.1,155.0,15/10/2023,88.7,II,250,Normal
126,P0112,38.0,F,69.1,158.6,19/09/2023,63.0,II,250,Normal
248,P0112,38.0,F,69.6,158.6,19/09/2023,63.0,II,250,Normal
82,P0137,62.0,F,46.4,148.8,19/08/2023,103.2,II,250,Anormal
312,P0137,62.0,F,47.5,148.8,"August 19, 2023",103.2,II,250,Anormal
470,P0190,62.0,M,76.9,169.7,18/07/2023,62.3,II,250,Normal
131,P0190,62.0,M,76.9,NaN,18/07/2023,62.3,II,250,Normal


In [48]:
# Verificar antes y depues de resolver duplicados por ID
print("Antes:", df_clean.shape)
df_clean = resolver_duplicados_por_id(df_clean, id_col="id_paciente")
print(f"Filas después de resolver duplicados por ID: {len(df_clean)}")


Antes: (515, 10)
Filas después de resolver duplicados por ID: 500


In [49]:
# Verificar que no queden IDs duplicados
print(
    "IDs duplicados restantes:",
    df_clean["id_paciente"].duplicated().sum()
)

IDs duplicados restantes: 0


## Corrección de edades fuera de rango

Se identificaron edades negativas y superiores a 120 años.

Estos valores se consideran inválidos y se reemplazan por valores nulos para posteriormente ser imputados.

In [50]:
mask = (
    (df_clean["edad_paciente"] < 0) |
    (df_clean["edad_paciente"] > 120)
)

df_clean.loc[
    mask,
    ["id_paciente", "edad_paciente"]
]

,id_paciente,edad_paciente
24,P0125,150.0
135,P0043,140.0
221,P0200,132.0
249,P0090,-3.0
262,P0147,-1.0
369,P0435,-5.0


In [51]:
df_clean = corregir_edades_imposibles(df_clean)

df_clean["edad_paciente"].describe()

count    490.000000
mean      54.655102
std       16.288222
min       18.000000
25%       44.000000
50%       56.000000
75%       65.000000
max       93.000000
Name: edad_paciente, dtype: float64

## Conversión de fechas

La variable `fecha_registro` se encontraba almacenada como texto.

Se convierte al tipo datetime para facilitar análisis posteriores y validar consistencia temporal.

In [67]:
# Aplicar normalización de fechas a la columna fecha_registro
df_clean = normalizar_fechas(df_clean)
df_clean["fecha_registro"].head()


0   2023-10-04
1   2023-04-23
2   2023-01-25
3   2023-06-24
4   2023-01-28
Name: fecha_registro, dtype: datetime64[us]

## Imputación de valores faltantes

Variables numéricas:
- edad_paciente
- peso_kg
- altura_cm

Se utiliza la mediana por ser robusta frente a valores extremos.

Variable categórica:
- sexo

Se utiliza la moda al representar la categoría más frecuente observada.

In [69]:
df_clean.isnull().sum()

id_paciente                       0
edad_paciente                    10
sexo                              4
peso_kg                           6
altura_cm                         5
fecha_registro                    0
frecuencia_cardiaca_media_bpm     0
derivacion_ecg                    0
frecuencia_muestreo_hz            0
etiqueta                          0
dtype: int64

In [70]:
df_clean = imputar_numericas_con_mediana(
    df_clean,
    [
        "edad_paciente",
        "peso_kg",
        "altura_cm"
    ]
)

In [71]:
df_clean = imputar_categorica_con_moda(
    df_clean,
    "sexo"
)

In [72]:
df_clean.isnull().sum()

id_paciente                      0
edad_paciente                    0
sexo                             0
peso_kg                          0
altura_cm                        0
fecha_registro                   0
frecuencia_cardiaca_media_bpm    0
derivacion_ecg                   0
frecuencia_muestreo_hz           0
etiqueta                         0
dtype: int64

## Verificación posterior a la limpieza

Se valida que:

- No existan valores faltantes.
- No existan edades fuera de rango.
- No existan identificadores duplicados.
- Las fechas hayan sido convertidas correctamente.

In [73]:
print("Registros:", len(df_clean))

print(
    "Nulos totales:",
    df_clean.isnull().sum().sum()
)

print(
    "IDs duplicados:",
    df_clean["id_paciente"].duplicated().sum()
)

print(
    "Edades inválidas:",
    (
        (df_clean["edad_paciente"] < 0)
        |
        (df_clean["edad_paciente"] > 120)
    ).sum()
)

Registros: 500
Nulos totales: 0
IDs duplicados: 0
Edades inválidas: 0


## Guardar el Dataset limpio

Se guarda la nueva version del dataset con los cambios de limpieza aplicados

In [74]:
df_clean.to_csv(
    "../data/clean-data/pacientes_clean.csv",
    index=False
)

print("Dataset limpio guardado correctamente.")

Dataset limpio guardado correctamente.
